In [1]:
# Import libraries

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import shap 
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset

data = pd.read_parquet("NF-BoT-IoT-V2.parquet")

print("Dataset shape:", data.shape)
print("Dataset columns:", data.columns.tolist())

# naming the label columns
label_column = "Attack"                     
drop_columns = ["Attack", "Label"]           

Dataset shape: (30420086, 43)
Dataset columns: ['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label', 'Attack']


In [3]:
# Data preprocessing

print("Data Preprocessing")

# remove duplicates
before = len(data)
data.drop_duplicates(inplace=True)
after = len(data)
print(f"Removed {before - after} duplicate rows.")

Data Preprocessing
Removed 0 duplicate rows.


In [4]:
# Drop the irrelevant identifier columns (avoids learning bias)
cols_to_drop = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT"]
cols_to_drop = [col for col in cols_to_drop if col in data.columns]
data.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped identifier columns: {cols_to_drop}")

Dropped identifier columns: ['L4_SRC_PORT', 'L4_DST_PORT']


In [5]:
# Check class distribution before balancing
print(f"\n Class distribution before balancing:\n{data[label_column].value_counts()}")


 Class distribution before balancing:
Attack
DDoS              14280259
DoS               13645057
Reconnaissance     2363017
Benign              129437
Theft                 2316
Name: count, dtype: int64


In [10]:
# sampling and balancing the dataset 

MAX_SAMPLES_PER_CLASS = 50_000

df_balanced = pd.concat([
    group.sample(n=min(len(group), MAX_SAMPLES_PER_CLASS), random_state=42)
    for name, group in data.groupby(label_column)
]).reset_index(drop=True)

print(f"\nClass distribution after balancing:\n{df_balanced[label_column].value_counts()}")
print(f"Balanced dataset shape: {df_balanced.shape}")


Class distribution after balancing:
Attack
Benign            50000
DDoS              50000
DoS               50000
Reconnaissance    50000
Theft              2316
Name: count, dtype: int64
Balanced dataset shape: (202316, 41)


In [14]:
# Split features and target
X = df_balanced.drop(columns=drop_columns)
y = df_balanced[label_column]

# Encode target labels
le  = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"\nEncoded class labels: {le.classes_}")


Encoded class labels: ['Benign' 'DDoS' 'DoS' 'Reconnaissance' 'Theft']


In [15]:
# Handle missing values

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)
print(f"Missing values after handling: {X.isnull().sum().sum()}")



Missing values after handling: 0


In [16]:
# Feature Scaling 
scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
print(f"Feature normalization applied (Min-Max Scaling)")

Feature normalization applied (Min-Max Scaling)


In [17]:
# Train test Split (80/20) 

print("Train-test split (80/20)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

Train-test split (80/20)
Training set shape: (161852, 39), (161852,)
Test set shape: (40464, 39), (40464,)


In [18]:
# Define and train the LightGBM model

model = lgb.LGBMClassifier(
    objective='multiclass', # multi-class classification
    n_estimators=300, # number of trees
    learning_rate=0.05, # learning rate
    num_leaves=64, # maximum number of leaves in one tree
    max_depth=-1, # no limit on tree depth
    min_child_samples=20, # minimum number of samples in a leaf
    subsample=0.8, # row sampling
    colsample_bytree=0.8, # feature sampling
    lambda_l1=0.1, # L1 regularization
    lambda_l2=0.1, # L2 regularization
    is_unbalance=True, # handle class imbalance
    n_jobs=-1, # use all CPU cores
    random_state=42 # for reproducibility
)

In [20]:
# 10-fold cross-validation

print("10-fold cross-validation")

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = cross_validate(
    model, X, y,
    cv=skf,
    scoring=[
        'accuracy',
        'precision_weighted',
        'recall_weighted',
        'f1_weighted',
        'roc_auc_ovr_weighted',
        'neg_log_loss'
    ],
    return_train_score=True,
    n_jobs=-1
)

print("\nCross-validation results:")
print(f"CV Accuracy:  {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"CV Precision: {cv_results['test_precision_weighted'].mean():.4f} ± {cv_results['test_precision_weighted'].std():.4f}")
print(f"CV Recall:    {cv_results['test_recall_weighted'].mean():.4f} ± {cv_results['test_recall_weighted'].std():.4f}")
print(f"CV F1 Score:  {cv_results['test_f1_weighted'].mean():.4f} ± {cv_results['test_f1_weighted'].std():.4f}")
print(f"CV ROC AUC:   {cv_results['test_roc_auc_ovr_weighted'].mean():.4f} ± {cv_results['test_roc_auc_ovr_weighted'].std():.4f}")
print(f"CV Log Loss:  {-cv_results['test_neg_log_loss'].mean():.4f} ± {cv_results['test_neg_log_loss'].std():.4f}")
 
# Per-fold results
print("\n--- Per-Fold Accuracy ---")
for i, acc in enumerate(cv_results['test_accuracy'], 1):
    print(f"  Fold {i:2d}: {acc:.4f}")

10-fold cross-validation

Cross-validation results:
CV Accuracy:  0.9822 ± 0.0011
CV Precision: 0.9823 ± 0.0011
CV Recall:    0.9822 ± 0.0011
CV F1 Score:  0.9822 ± 0.0011
CV ROC AUC:   0.9991 ± 0.0001
CV Log Loss:  0.0508 ± 0.0022

--- Per-Fold Accuracy ---
  Fold  1: 0.9811
  Fold  2: 0.9837
  Fold  3: 0.9817
  Fold  4: 0.9804
  Fold  5: 0.9813
  Fold  6: 0.9831
  Fold  7: 0.9836
  Fold  8: 0.9825
  Fold  9: 0.9812
  Fold 10: 0.9832


In [23]:
# Train final model on the entire training set

print("TRAINING FINAL MODEL (80/20 SPLIT FOR PLOTS)")

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=True),
        lgb.log_evaluation(period=25)
    ]
)
 
print(f"\nBest iteration: {model.best_iteration_}")
 
# Save model
model.booster_.save_model('lightgbm_model.txt')
print("Model saved to 'lightgbm_model.txt'")

TRAINING FINAL MODEL (80/20 SPLIT FOR PLOTS)
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004662 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5552
[LightGBM] [Info] Number of data points in the train set: 161852, number of used features: 38
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current v